# Week 10: Offline Evaluation and Comparison

This notebook evaluates all three recommenders against a held-out interaction protocol and produces
the Week 10 evaluation report.

Goal for this step:
- define a rigorous offline evaluation protocol using leave-one-out (LOO) on user histories
- evaluate all three systems: popularity_global, content_cosine, svd_collaborative
- compute Precision@K, Recall@K, NDCG@K, and Hit Rate@K for K ∈ {5, 10, 20}
- identify strong and failure cases for each system
- save the evaluation table and error analysis

Evaluation type: **item-to-item ranking** (given a query movie a user rated, rank the candidates
and check whether the held-out movie appears in top-K).
This is appropriate because our models produce movie-level scores, not user-level predictions.

## Evaluation protocol

**Candidate pool**: For each query movie in the top-5,000 most-rated movies, the candidate
pool is the precomputed top-20 recommendations from each system.

**Relevance proxy**: A movie is considered relevant to a query if it shares at least one genre.
This is a content-based relevance proxy that does not require user-level ground truth,
making it reproducible across the full catalog.

**Why item-item evaluation**: We built item-to-item recommenders (not user-to-item).
The correct evaluation is: given a query movie, do the top-K recommendations share
the same genre family? This is the standard evaluation for catalog discovery systems.

**Metrics**:
- **Precision@K**: fraction of top-K recommendations that are genre-relevant
- **Recall@K**: fraction of all genre-relevant movies in the top-K pool
- **NDCG@K**: normalized discounted cumulative gain (position-weighted relevance)
- **Hit Rate@K**: fraction of queries with at least one relevant recommendation in top-K

---

> **Actualización (LOO):** Se añadió una segunda capa de evaluación —
> **Leave-One-Out (LOO)** — en las secciones 11–15.  Para cada usuario se
> oculta su último ítem (ordenado por timestamp) y se mide si los tres sistemas
> lo recuperan en el top-K.  La métrica central en LOO es **Hit Rate@K**
> (equivalente a Recall@K cuando hay exactamente 1 ítem de test).
>
> El Recall@K de la evaluación por género tenía un defecto: su denominador era
> el pool candidato del sistema, no el universo real.  Queda documentado en la
> función `recall_at_k` y el LOO lo reemplaza como métrica de recall correcta.


In [1]:
from pathlib import Path

import json
import math

import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
if not (project_root / 'data').exists():
    project_root = project_root.parent

ARTIFACTS_DIR = project_root / 'artifacts' / 'week10'
WEEK07_DIR = project_root / 'artifacts' / 'week07'
DATA_DIR = project_root / 'data' / 'processed' / 'week03_v1'
RANDOM_STATE = 42

required = [
    ARTIFACTS_DIR / 'week10_content_recs_top20.parquet',
    ARTIFACTS_DIR / 'week10_svd_recs_top20.parquet',
    ARTIFACTS_DIR / 'week10_popularity_global.csv',
    ARTIFACTS_DIR / 'week10_baseline_meta.json',
    DATA_DIR / 'movies_catalog.parquet',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing required inputs: {missing}')

ARTIFACTS_DIR

PosixPath('/Users/jay17/Documents/Proyects/big-data-tf/artifacts/week10')

## 1) Load all recommendation outputs

Load the three recommendation sets from Notebooks 1 and 2.

In [2]:
# Load catalog for genre information
catalog_pl = pl.read_parquet(DATA_DIR / 'movies_catalog.parquet')
genre_map = dict(zip(catalog_pl['movieId'].to_list(), catalog_pl['genres_list'].to_list()))
title_map = dict(zip(catalog_pl['movieId'].to_list(), catalog_pl['title'].to_list()))

# Load recommendation sets
content_recs = pl.read_parquet(ARTIFACTS_DIR / 'week10_content_recs_top20.parquet')
svd_recs = pl.read_parquet(ARTIFACTS_DIR / 'week10_svd_recs_top20.parquet')
popularity_global = pl.read_csv(ARTIFACTS_DIR / 'week10_popularity_global.csv')

# Baseline meta
with open(ARTIFACTS_DIR / 'week10_baseline_meta.json') as f:
    baseline_meta = json.load(f)

# Get the common set of query movies (intersection of content and SVD)
content_query_ids = set(content_recs['query_movieId'].unique().to_list())
svd_query_ids = set(svd_recs['query_movieId'].unique().to_list())
common_query_ids = sorted(content_query_ids & svd_query_ids)

print(f'Content recs: {content_recs.height:,} rows, {len(content_query_ids):,} query movies')
print(f'SVD recs:     {svd_recs.height:,} rows, {len(svd_query_ids):,} query movies')
print(f'Common query movies for evaluation: {len(common_query_ids):,}')

Content recs: 100,000 rows, 5,000 query movies
SVD recs:     100,000 rows, 5,000 query movies
Common query movies for evaluation: 4,999


## 2) Build genre-relevance lookup

For each movie, precompute the set of genres. Two movies are relevant if they share at least one genre.
This is the ground-truth relevance signal used in our evaluation.

In [3]:
def get_genres(movie_id: int) -> set:
    """Return the set of genres for a movie, or empty set if unknown."""
    genres = genre_map.get(movie_id, None)
    if genres is None:
        return set()
    if isinstance(genres, list):
        return set(genres)
    return set(str(genres).split('|'))


def is_genre_relevant(query_id: int, rec_id: int) -> bool:
    """Return True if query and recommendation share at least one genre."""
    q_genres = get_genres(query_id)
    r_genres = get_genres(rec_id)
    if not q_genres or not r_genres:
        return False
    return len(q_genres & r_genres) > 0


# Quick check
print('Genre relevance examples:')
test_pairs = [(1, 2), (1, 296), (318, 593)]
for q, r in test_pairs:
    rel = is_genre_relevant(q, r)
    print(f'  {title_map.get(q, q)!r} → {title_map.get(r, r)!r}: genres_shared={rel}')

Genre relevance examples:
  'Toy Story (1995)' → 'Jumanji (1995)': genres_shared=True
  'Toy Story (1995)' → 'Pulp Fiction (1994)': genres_shared=True
  'Shawshank Redemption, The (1994)' → 'Silence of the Lambs, The (1991)': genres_shared=True


## 3) Evaluation functions

Implementation of Precision@K, Recall@K, NDCG@K, and Hit Rate@K.

In [4]:
def precision_at_k(relevant_flags: list[bool], k: int) -> float:
    """Fraction of top-K items that are relevant."""
    top_k = relevant_flags[:k]
    return sum(top_k) / k if k > 0 else 0.0


def recall_at_k(relevant_flags: list[bool], total_relevant: int, k: int) -> float:
    """Fraction of all relevant items retrieved in top-K.

    NOTE (corrección): cuando se usa evaluación item-to-item por género,
    el `total_relevant` debe ser el número de ítems relevantes en el universo
    completo del catálogo — NO en la lista candidata del sistema.  Usar el
    pool candidato como denominador hace que el Recall dependa del output del
    propio sistema, sesgando la métrica al alza.

    En la evaluación Leave-One-Out (sección 11 en adelante), este problema
    no aplica porque hay exactamente 1 ítem de test por usuario; en ese caso
    Recall@K ≡ Hit Rate@K y se reporta como tal.
    """
    if total_relevant == 0:
        return 0.0
    return sum(relevant_flags[:k]) / total_relevant


def dcg_at_k(relevant_flags: list[bool], k: int) -> float:
    """Discounted cumulative gain at K."""
    dcg = 0.0
    for i, rel in enumerate(relevant_flags[:k], 1):
        dcg += (2 ** int(rel) - 1) / math.log2(i + 1)
    return dcg


def ndcg_at_k(relevant_flags: list[bool], k: int) -> float:
    """Normalized DCG at K (ideal DCG = all relevant items at top positions)."""
    n_relevant = sum(relevant_flags[:k])
    ideal_flags = [True] * n_relevant + [False] * (k - n_relevant)
    ideal_dcg = dcg_at_k(ideal_flags, k)
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(relevant_flags, k) / ideal_dcg


def hit_rate_at_k(relevant_flags: list[bool], k: int) -> float:
    """1 if at least one item in top-K is relevant, else 0."""
    return 1.0 if any(relevant_flags[:k]) else 0.0


def evaluate_system(rec_dict: dict[int, list[int]], k_values: list[int]) -> dict:
    """Evaluate a recommendation system.

    Args:
        rec_dict: {query_movieId: [rec_movieId_rank1, ..., rec_movieId_rankN]}
        k_values: list of K thresholds to evaluate at

    Returns:
        Dictionary of metric averages.
    """
    results = {k: {'precision': [], 'recall': [], 'ndcg': [], 'hit_rate': []} for k in k_values}

    for query_id, rec_ids in rec_dict.items():
        q_genres = get_genres(query_id)
        if not q_genres:
            continue
        # How many items in the candidate pool are relevant?
        relevant_flags = [is_genre_relevant(query_id, rid) for rid in rec_ids]
        total_relevant = sum(relevant_flags)

        for k in k_values:
            results[k]['precision'].append(precision_at_k(relevant_flags, k))
            results[k]['recall'].append(recall_at_k(relevant_flags, total_relevant, k))
            results[k]['ndcg'].append(ndcg_at_k(relevant_flags, k))
            results[k]['hit_rate'].append(hit_rate_at_k(relevant_flags, k))

    return {
        k: {
            'precision': np.mean(v['precision']),
            'recall': np.mean(v['recall']),
            'ndcg': np.mean(v['ndcg']),
            'hit_rate': np.mean(v['hit_rate']),
            'n_queries': len(v['precision']),
        }
        for k, v in results.items()
    }

## 4) Evaluate all three systems

Build recommendation dictionaries for each system and run the evaluation loop.

In [5]:
K_VALUES = [5, 10, 20]
TOP_N_POP = 20  # how many popularity recommendations to use

# Build rec dicts (query_id → ordered list of rec_ids)

# --- Content-based ---
content_dict: dict[int, list[int]] = {}
for row in content_recs.filter(pl.col('query_movieId').is_in(common_query_ids)).sort(['query_movieId', 'rank']).iter_rows(named=True):
    content_dict.setdefault(row['query_movieId'], []).append(row['rec_movieId'])

# --- SVD collaborative ---
svd_dict: dict[int, list[int]] = {}
for row in svd_recs.filter(pl.col('query_movieId').is_in(common_query_ids)).sort(['query_movieId', 'rank']).iter_rows(named=True):
    svd_dict.setdefault(row['query_movieId'], []).append(row['rec_movieId'])

# --- Popularity global ---
# For popularity, top-N globally regardless of query
pop_top_ids = popularity_global.sort('rating_count', descending=True).head(TOP_N_POP)['movieId'].to_list()
pop_dict: dict[int, list[int]] = {
    qid: [mid for mid in pop_top_ids if mid != qid][:TOP_N_POP]
    for qid in common_query_ids
}

print(f'Content queries: {len(content_dict):,}')
print(f'SVD queries:     {len(svd_dict):,}')
print(f'Pop queries:     {len(pop_dict):,}')
print(f'Evaluating at K = {K_VALUES}...')

results_content = evaluate_system(content_dict, K_VALUES)
results_svd     = evaluate_system(svd_dict, K_VALUES)
results_pop     = evaluate_system(pop_dict, K_VALUES)

print('Evaluation complete.')

Content queries: 4,999
SVD queries:     4,999
Pop queries:     4,999
Evaluating at K = [5, 10, 20]...
Evaluation complete.


## 5) Evaluation results table

Aggregate the metrics into a single comparison table. Each row is one (model, K) combination.

In [6]:
eval_rows = []
for system_name, results in [
    ('popularity_global', results_pop),
    ('content_cosine', results_content),
    ('svd_collaborative', results_svd),
]:
    for k in K_VALUES:
        m = results[k]
        eval_rows.append({
            'system': system_name,
            'k': k,
            'precision_at_k': round(m['precision'], 4),
            'recall_at_k': round(m['recall'], 4),
            'ndcg_at_k': round(m['ndcg'], 4),
            'hit_rate_at_k': round(m['hit_rate'], 4),
            'n_queries': m['n_queries'],
        })

eval_df = pl.DataFrame(eval_rows)
eval_df.write_csv(ARTIFACTS_DIR / 'week10_evaluation_results.csv')

print('Evaluation results:')
display(eval_df.to_pandas().to_string(index=False))

Evaluation results:


'           system  k  precision_at_k  recall_at_k  ndcg_at_k  hit_rate_at_k  n_queries\npopularity_global  5          0.5820       0.3621     0.8437         0.9553       4994\npopularity_global 10          0.5554       0.6089     0.8390         0.9720       4994\npopularity_global 20          0.4593       0.9808     0.8206         0.9808       4994\n   content_cosine  5          0.9822       0.2535     0.9930         0.9982       4994\n   content_cosine 10          0.9775       0.5032     0.9925         0.9990       4994\n   content_cosine 20          0.9722       0.9994     0.9916         0.9994       4994\nsvd_collaborative  5          0.8807       0.2656     0.9525         0.9890       4994\nsvd_collaborative 10          0.8603       0.5155     0.9496         0.9958       4994\nsvd_collaborative 20          0.8379       0.9986     0.9462         0.9986       4994'

## 6) Evaluation plots

Four grouped bar charts comparing all three systems across K values for each metric.

In [7]:
systems = ['popularity_global', 'content_cosine', 'svd_collaborative']
system_colors = {'popularity_global': '#10b981', 'content_cosine': '#6366f1', 'svd_collaborative': '#f59e0b'}
system_labels = {'popularity_global': 'Popularity', 'content_cosine': 'Content (cosine)', 'svd_collaborative': 'SVD (collab.)'}

metrics = [
    ('precision_at_k', 'Precision@K'),
    ('recall_at_k', 'Recall@K'),
    ('ndcg_at_k', 'NDCG@K'),
    ('hit_rate_at_k', 'Hit Rate@K'),
]

fig_eval = make_subplots(
    rows=2, cols=2,
    subplot_titles=[m[1] for m in metrics],
)

positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
k_labels = [str(k) for k in K_VALUES]

for (metric_col, metric_title), (row, col) in zip(metrics, positions):
    for system in systems:
        sys_data = eval_df.filter(pl.col('system') == system).sort('k')
        fig_eval.add_trace(
            go.Bar(
                name=system_labels[system],
                x=k_labels,
                y=sys_data[metric_col].to_list(),
                marker_color=system_colors[system],
                showlegend=(row == 1 and col == 1),
            ),
            row=row, col=col,
        )

fig_eval.update_layout(
    title='Offline evaluation: all three systems at K ∈ {5, 10, 20}',
    height=700,
    template='plotly_white',
    barmode='group',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
for row, col in positions:
    fig_eval.update_xaxes(title_text='K', row=row, col=col)

fig_eval.write_html(ARTIFACTS_DIR / 'week10_evaluation_comparison.html')
fig_eval.write_image(ARTIFACTS_DIR / 'week10_evaluation_comparison.png', scale=2)
fig_eval.show()

## 7) NDCG@10 per-system bar chart (main result)

Single-metric summary chart for the milestone report.

In [8]:
ndcg10 = (
    eval_df
    .filter(pl.col('k') == 10)
    .select(['system', 'ndcg_at_k'])
    .sort('ndcg_at_k', descending=True)
)

fig_ndcg = go.Figure(go.Bar(
    x=[system_labels.get(s, s) for s in ndcg10['system'].to_list()],
    y=ndcg10['ndcg_at_k'].to_list(),
    marker_color=[system_colors.get(s, '#888') for s in ndcg10['system'].to_list()],
    text=[f'{v:.4f}' for v in ndcg10['ndcg_at_k'].to_list()],
    textposition='outside',
))
fig_ndcg.update_layout(
    title='NDCG@10 comparison across recommendation systems',
    xaxis_title='System',
    yaxis_title='NDCG@10',
    height=600,
    template='plotly_white',
    yaxis_range=[0, ndcg10['ndcg_at_k'].max() * 1.25],
)
fig_ndcg.write_html(ARTIFACTS_DIR / 'week10_ndcg10_comparison.html')
fig_ndcg.write_image(ARTIFACTS_DIR / 'week10_ndcg10_comparison.png', scale=2)
fig_ndcg.show()

## 8) Error analysis — strong and failure cases

For each system we identify:
- **Strong cases**: queries where all 10 top recommendations are genre-relevant (Precision@10 = 1.0)
- **Failure cases**: queries where none of the top 10 recommendations are relevant (Hit Rate@10 = 0)

This qualitative analysis reveals structural weaknesses in each approach.

In [9]:
def compute_per_query_metrics(rec_dict: dict[int, list[int]], k: int = 10) -> pl.DataFrame:
    """Compute per-query precision@K for error analysis."""
    rows = []
    for qid, recs in rec_dict.items():
        flags = [is_genre_relevant(qid, rid) for rid in recs]
        p_at_k = precision_at_k(flags, k)
        h_at_k = hit_rate_at_k(flags, k)
        rows.append({
            'query_movieId': qid,
            'query_title': title_map.get(qid, '?'),
            'query_genres': '|'.join(sorted(get_genres(qid))),
            'precision_at_k': round(p_at_k, 4),
            'hit_rate_at_k': round(h_at_k, 4),
        })
    return pl.DataFrame(rows)


ANALYSIS_K = 10
error_analysis_rows = []

for system_name, rec_dict in [
    ('popularity_global', pop_dict),
    ('content_cosine', content_dict),
    ('svd_collaborative', svd_dict),
]:
    per_query = compute_per_query_metrics(rec_dict, k=ANALYSIS_K)

    strong = per_query.filter(pl.col('precision_at_k') == 1.0).head(5)
    failures = per_query.filter(pl.col('hit_rate_at_k') == 0.0).head(5)

    for row in strong.iter_rows(named=True):
        error_analysis_rows.append({'system': system_name, 'case_type': 'strong', **row})
    for row in failures.iter_rows(named=True):
        error_analysis_rows.append({'system': system_name, 'case_type': 'failure', **row})

error_df = pl.DataFrame(error_analysis_rows)
error_df.write_csv(ARTIFACTS_DIR / 'week10_error_analysis.csv')

print('Error analysis (strong and failure cases per system):')
display(error_df.to_pandas())

Error analysis (strong and failure cases per system):


,system,case_type,query_movieId,query_title,query_genres,precision_at_k,hit_rate_at_k
0,popularity_global,strong,20,Money Train (1995),Action|Comedy|Crime|Drama|Thriller,1.0,1.0
1,popularity_global,strong,42,Dead Presidents (1995),Action|Crime|Drama,1.0,1.0
2,popularity_global,strong,78,"Crossing Guard, The (1995)",Action|Crime|Drama|Thriller,1.0,1.0
3,popularity_global,strong,145,Bad Boys (1995),Action|Comedy|Crime|Drama|Thriller,1.0,1.0
4,popularity_global,strong,198,Strange Days (1995),Action|Crime|Drama|Mystery|Sci-Fi|Thriller,1.0,1.0
5,popularity_global,failure,99,Heidi Fleiss: Hollywood Madam (1995),Documentary,0.0,0.0
6,popularity_global,failure,116,Anne Frank Remembered (1995),Documentary,0.0,0.0
7,popularity_global,failure,162,Crumb (1994),Documentary,0.0,0.0
8,popularity_global,failure,206,Unzipped (1995),Documentary,0.0,0.0
9,popularity_global,failure,246,Hoop Dreams (1994),Documentary,0.0,0.0


## 9) Per-query NDCG@10 distribution

Box plots of per-query NDCG@10 for each system, showing median and spread of recommendation quality.

In [10]:
box_data = []
for system_name, rec_dict in [
    ('popularity_global', pop_dict),
    ('content_cosine', content_dict),
    ('svd_collaborative', svd_dict),
]:
    for qid, recs in rec_dict.items():
        flags = [is_genre_relevant(qid, rid) for rid in recs]
        ndcg = ndcg_at_k(flags, 10)
        box_data.append({'system': system_name, 'ndcg_at_10': ndcg})

box_pl = pl.DataFrame(box_data)

fig_box = go.Figure()
for system in systems:
    vals = box_pl.filter(pl.col('system') == system)['ndcg_at_10'].to_list()
    fig_box.add_trace(go.Box(
        y=vals,
        name=system_labels[system],
        marker_color=system_colors[system],
        boxmean='sd',
    ))

fig_box.update_layout(
    title='Per-query NDCG@10 distribution by system',
    yaxis_title='NDCG@10',
    height=480,
    template='plotly_white',
)
fig_box.write_html(ARTIFACTS_DIR / 'week10_ndcg10_distribution.html')
fig_box.write_image(ARTIFACTS_DIR / 'week10_ndcg10_distribution.png', scale=2)
fig_box.show()

## 10) Save evaluation summary JSON

In [11]:
eval_summary = {
    'evaluation_type': 'item_to_item_genre_relevance',
    'relevance_criterion': 'shared_at_least_one_genre',
    'k_values': K_VALUES,
    'n_query_movies': len(common_query_ids),
    'results': {
        system: {
            f'k_{k}': {
                'precision': round(results[k]['precision'], 6),
                'recall': round(results[k]['recall'], 6),
                'ndcg': round(results[k]['ndcg'], 6),
                'hit_rate': round(results[k]['hit_rate'], 6),
            }
            for k in K_VALUES
        }
        for system, results in [
            ('popularity_global', results_pop),
            ('content_cosine', results_content),
            ('svd_collaborative', results_svd),
        ]
    },
    'artifacts': [
        'week10_evaluation_results.csv',
        'week10_error_analysis.csv',
        'week10_evaluation_comparison.html',
        'week10_ndcg10_comparison.html',
        'week10_ndcg10_distribution.html',
    ],
}

with open(ARTIFACTS_DIR / 'week10_evaluation_summary.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)

print('Evaluation summary saved.')
print(json.dumps(eval_summary, indent=2))

Evaluation summary saved.
{
  "evaluation_type": "item_to_item_genre_relevance",
  "relevance_criterion": "shared_at_least_one_genre",
  "k_values": [
    5,
    10,
    20
  ],
  "n_query_movies": 4999,
  "results": {
    "popularity_global": {
      "k_5": {
        "precision": 0.582018,
        "recall": 0.36215,
        "ndcg": 0.843694,
        "hit_rate": 0.955346
      },
      "k_10": {
        "precision": 0.555447,
        "recall": 0.608891,
        "ndcg": 0.839018,
        "hit_rate": 0.971966
      },
      "k_20": {
        "precision": 0.459331,
        "recall": 0.980777,
        "ndcg": 0.820592,
        "hit_rate": 0.980777
      }
    },
    "content_cosine": {
      "k_5": {
        "precision": 0.982179,
        "recall": 0.253453,
        "ndcg": 0.993042,
        "hit_rate": 0.998198
      },
      "k_10": {
        "precision": 0.977493,
        "recall": 0.503199,
        "ndcg": 0.992487,
        "hit_rate": 0.998999
      },
      "k_20": {
        "precisi

## 11) Leave-One-Out (LOO) Evaluation

La evaluación por géneros mide *relevancia de catálogo* (¿los ítems recomendados
comparten género?), pero no mide si el sistema predice correctamente el próximo
consumo del usuario.

**Protocolo LOO**
- Cargar todos los ratings con timestamp.
- Filtrar usuarios con **≥ 5 ratings**.
- Por usuario: ocultar el **último ítem** (el más reciente) como test; el resto
  es entrenamiento.
- Medir si el ítem oculto aparece en el top-K generado por cada sistema.

**Métricas**
- **Hit Rate@K** (= Recall@K en LOO): ¿el ítem oculto está en top-K?
- **Precision@K**: fracción de top-K que son relevantes (para LOO = 1/K si hay hit).
- **NDCG@K**: hit ponderado por posición.

> Se usa una **muestra aleatoria de 10 000 usuarios** para mantener el tiempo
> de cómputo razonable (seed=42, reproducible).


In [12]:
import random

LOO_SAMPLE_USERS = 10_000
LOO_SEED = 42
K_VALUES_LOO = [5, 10, 20]

# Load ratings
ratings_all = pl.read_parquet(DATA_DIR / 'ratings_clean.parquet')

# Keep users with >= 5 ratings
user_counts = ratings_all.group_by('userId').agg(pl.len().alias('n'))
active_user_ids = user_counts.filter(pl.col('n') >= 5)['userId'].to_list()
print(f'Users with >=5 ratings: {len(active_user_ids):,}')

# Subsample for speed
random.seed(LOO_SEED)
sampled_user_ids = random.sample(active_user_ids, min(LOO_SAMPLE_USERS, len(active_user_ids)))
print(f'Sampled users: {len(sampled_user_ids):,}')

# Filter and rank by timestamp (rank 1 = most recent)
ratings_sampled = ratings_all.filter(pl.col('userId').is_in(sampled_user_ids))
ratings_ranked = ratings_sampled.with_columns(
    pl.col('timestamp')
    .rank(method='ordinal', descending=True)
    .over('userId')
    .alias('rank_desc')
)

loo_test  = (ratings_ranked
             .filter(pl.col('rank_desc') == 1)
             .select(['userId', 'movieId'])
             .rename({'movieId': 'test_movieId'}))

loo_train = (ratings_ranked
             .filter(pl.col('rank_desc') != 1)
             .select(['userId', 'movieId', 'rating']))

print(f'LOO test set:  {loo_test.height:,} users')
print(f'LOO train set: {loo_train.height:,} ratings')
print(loo_test.head(3))


Users with >=5 ratings: 162,541
Sampled users: 10,000
LOO test set:  10,000 users
LOO train set: 1,499,978 ratings
shape: (3, 2)
┌────────┬──────────────┐
│ userId ┆ test_movieId │
│ ---    ┆ ---          │
│ i64    ┆ i64          │
╞════════╪══════════════╡
│ 52     ┆ 2329         │
│ 65     ┆ 73321        │
│ 74     ┆ 10           │
└────────┴──────────────┘


### Popularity LOO

In [13]:
# --- LOO: Popularity baseline ---
# Top-500 most-rated items in train; exclude items already seen by the user.

POP_POOL = 500

pop_train_counts = (loo_train
                    .group_by('movieId')
                    .agg(pl.len().alias('n_ratings'))
                    .sort('n_ratings', descending=True))
pop_top_ids_loo = pop_train_counts.head(POP_POOL)['movieId'].to_list()

# Build per-user seen set (from train)
user_seen_map: dict[int, set] = {}
for row in loo_train.iter_rows(named=True):
    user_seen_map.setdefault(row['userId'], set()).add(row['movieId'])

# Evaluate
loo_pop_hits  = {k: 0 for k in K_VALUES_LOO}
loo_pop_ndcg  = {k: 0.0 for k in K_VALUES_LOO}
loo_pop_prec  = {k: 0.0 for k in K_VALUES_LOO}
loo_n_users = 0

for row in loo_test.iter_rows(named=True):
    uid, tid = row['userId'], row['test_movieId']
    seen = user_seen_map.get(uid, set())
    recs = [m for m in pop_top_ids_loo if m not in seen]
    for k in K_VALUES_LOO:
        top_k = recs[:k]
        hit = tid in top_k
        loo_pop_hits[k] += int(hit)
        loo_pop_prec[k] += (1 / k) if hit else 0.0
        if hit:
            pos = top_k.index(tid) + 1
            loo_pop_ndcg[k] += 1.0 / math.log2(pos + 1)
    loo_n_users += 1

print(f'Popularity LOO (n={loo_n_users:,}):')
for k in K_VALUES_LOO:
    print(f'  Hit Rate@{k}: {loo_pop_hits[k]/loo_n_users:.4f} | '
          f'Precision@{k}: {loo_pop_prec[k]/loo_n_users:.4f} | '
          f'NDCG@{k}: {loo_pop_ndcg[k]/loo_n_users:.4f}')


Popularity LOO (n=10,000):
  Hit Rate@5: 0.0287 | Precision@5: 0.0057 | NDCG@5: 0.0180
  Hit Rate@10: 0.0465 | Precision@10: 0.0047 | NDCG@10: 0.0238
  Hit Rate@20: 0.0767 | Precision@20: 0.0038 | NDCG@20: 0.0313


### Content-based LOO (SVD item embeddings)

In [14]:
item_factors = pl.read_parquet(ARTIFACTS_DIR / 'week10_svd_item_factors.parquet')
# --- LOO: Content-based (SVD item factors as item embedding) ---
# User profile = weighted average of train item vectors.
# Score = cosine similarity between profile and each candidate item.

factor_cols_loo = [c for c in item_factors.columns if c.startswith('svd_')]
item_ids_loo = item_factors['movieId'].to_list()
item_matrix_loo = item_factors.select(factor_cols_loo).to_numpy()
item_id_to_idx_loo = {mid: i for i, mid in enumerate(item_ids_loo)}
item_id_set_loo = set(item_ids_loo)

# L2-normalise
_norms = np.linalg.norm(item_matrix_loo, axis=1, keepdims=True)
_norms[_norms == 0] = 1
item_matrix_norm_loo = item_matrix_loo / _norms

# Build user train histories (items with known factors only)
user_history_loo: dict[int, list[tuple[int, float]]] = {}
for row in loo_train.iter_rows(named=True):
    mid = row['movieId']
    if mid in item_id_to_idx_loo:
        user_history_loo.setdefault(row['userId'], []).append((mid, row['rating']))

# Evaluate
loo_cb_hits  = {k: 0 for k in K_VALUES_LOO}
loo_cb_ndcg  = {k: 0.0 for k in K_VALUES_LOO}
loo_cb_prec  = {k: 0.0 for k in K_VALUES_LOO}
loo_cb_n = 0

for row in loo_test.iter_rows(named=True):
    uid, tid = row['userId'], row['test_movieId']
    if tid not in item_id_set_loo:
        continue
    history = user_history_loo.get(uid, [])
    if not history:
        continue
    seen = {m for m, _ in history}
    weights = np.array([r for _, r in history], dtype=np.float32)
    vecs = np.stack([item_matrix_norm_loo[item_id_to_idx_loo[m]] for m, _ in history])
    profile = (vecs * weights[:, None]).sum(axis=0)
    pnorm = np.linalg.norm(profile)
    if pnorm < 1e-9:
        continue
    profile /= pnorm
    scores = item_matrix_norm_loo @ profile
    # sort candidates (exclude seen)
    cand_idx = [i for i, m in enumerate(item_ids_loo) if m not in seen]
    cand_sorted = sorted(cand_idx, key=lambda i: -scores[i])
    top_ids = [item_ids_loo[i] for i in cand_sorted]
    for k in K_VALUES_LOO:
        top_k = top_ids[:k]
        hit = tid in top_k
        loo_cb_hits[k] += int(hit)
        loo_cb_prec[k] += (1 / k) if hit else 0.0
        if hit:
            pos = top_k.index(tid) + 1
            loo_cb_ndcg[k] += 1.0 / math.log2(pos + 1)
    loo_cb_n += 1

print(f'Content-based LOO (n={loo_cb_n:,}):')
for k in K_VALUES_LOO:
    print(f'  Hit Rate@{k}: {loo_cb_hits[k]/loo_cb_n:.4f} | '
          f'Precision@{k}: {loo_cb_prec[k]/loo_cb_n:.4f} | '
          f'NDCG@{k}: {loo_cb_ndcg[k]/loo_cb_n:.4f}')


Content-based LOO (n=9,885):
  Hit Rate@5: 0.0219 | Precision@5: 0.0044 | NDCG@5: 0.0139
  Hit Rate@10: 0.0368 | Precision@10: 0.0037 | NDCG@10: 0.0187
  Hit Rate@20: 0.0563 | Precision@20: 0.0028 | NDCG@20: 0.0236


### SVD Collaborative LOO

In [15]:
# --- LOO: SVD collaborative (item-factor dot-product score) ---
# For each user, compute a user vector = mean of train item factors (unweighted).
# Score = dot product between user vector and each candidate item factor.
# (Approximates the standard SVD user-item score without re-fitting.)

loo_svd_hits  = {k: 0 for k in K_VALUES_LOO}
loo_svd_ndcg  = {k: 0.0 for k in K_VALUES_LOO}
loo_svd_prec  = {k: 0.0 for k in K_VALUES_LOO}
loo_svd_n = 0

for row in loo_test.iter_rows(named=True):
    uid, tid = row['userId'], row['test_movieId']
    if tid not in item_id_to_idx_loo:
        continue
    history = user_history_loo.get(uid, [])
    if not history:
        continue
    seen = {m for m, _ in history}
    # unweighted mean of raw (non-normalised) item factors as user vector
    vecs_raw = np.stack([item_matrix_loo[item_id_to_idx_loo[m]] for m, _ in history])
    user_vec = vecs_raw.mean(axis=0)
    scores = item_matrix_loo @ user_vec
    cand_idx = [i for i, m in enumerate(item_ids_loo) if m not in seen]
    cand_sorted = sorted(cand_idx, key=lambda i: -scores[i])
    top_ids = [item_ids_loo[i] for i in cand_sorted]
    for k in K_VALUES_LOO:
        top_k = top_ids[:k]
        hit = tid in top_k
        loo_svd_hits[k] += int(hit)
        loo_svd_prec[k] += (1 / k) if hit else 0.0
        if hit:
            pos = top_k.index(tid) + 1
            loo_svd_ndcg[k] += 1.0 / math.log2(pos + 1)
    loo_svd_n += 1

print(f'SVD collaborative LOO (n={loo_svd_n:,}):')
for k in K_VALUES_LOO:
    print(f'  Hit Rate@{k}: {loo_svd_hits[k]/loo_svd_n:.4f} | '
          f'Precision@{k}: {loo_svd_prec[k]/loo_svd_n:.4f} | '
          f'NDCG@{k}: {loo_svd_ndcg[k]/loo_svd_n:.4f}')


SVD collaborative LOO (n=9,885):
  Hit Rate@5: 0.0515 | Precision@5: 0.0103 | NDCG@5: 0.0343
  Hit Rate@10: 0.0795 | Precision@10: 0.0080 | NDCG@10: 0.0432
  Hit Rate@20: 0.1221 | Precision@20: 0.0061 | NDCG@20: 0.0538


### LOO Results table and chart

In [19]:
# --- LOO results table ---
loo_rows = []
for sys_name, hits, prec, ndcg, n_q in [
    ('popularity_global', loo_pop_hits, loo_pop_prec, loo_pop_ndcg, loo_n_users),
    ('content_cosine',    loo_cb_hits,  loo_cb_prec,  loo_cb_ndcg,  loo_cb_n),
    ('svd_collaborative', loo_svd_hits, loo_svd_prec, loo_svd_ndcg, loo_svd_n),
]:
    for k in K_VALUES_LOO:
        loo_rows.append({
            'system': sys_name,
            'k': k,
            'hit_rate_at_k':  round(hits[k] / n_q, 4),
            'precision_at_k': round(prec[k] / n_q, 4),
            'ndcg_at_k':      round(ndcg[k] / n_q, 4),
            'n_queries': n_q,
        })

loo_df = pl.DataFrame(loo_rows)
loo_df.write_csv(ARTIFACTS_DIR / 'week10_loo_evaluation_results.csv')
print('LOO evaluation results:')
display(loo_df.to_pandas().to_string(index=False))

# Bar chart comparison
fig_loo = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Hit Rate@K (LOO)', 'Precision@K (LOO)', 'NDCG@K (LOO)'],
)
loo_colors = {'popularity_global': '#10b981', 'content_cosine': '#6366f1', 'svd_collaborative': '#f59e0b'}
loo_labels = {'popularity_global': 'Popularity', 'content_cosine': 'Content (cosine)', 'svd_collaborative': 'SVD (collab.)'}
k_labels_loo = [str(k) for k in K_VALUES_LOO]

for col_idx, (metric_col, metric_name) in enumerate(
    [('hit_rate_at_k', 'Hit Rate'), ('precision_at_k', 'Precision'), ('ndcg_at_k', 'NDCG')], 1
):
    for sys_name in ['popularity_global', 'content_cosine', 'svd_collaborative']:
        sys_data = loo_df.filter(pl.col('system') == sys_name).sort('k')
        fig_loo.add_trace(
            go.Bar(
                name=loo_labels[sys_name],
                x=k_labels_loo,
                y=sys_data[metric_col].to_list(),
                marker_color=loo_colors[sys_name],
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )
        fig_loo.update_xaxes(title_text='K', row=1, col=col_idx)

fig_loo.update_layout(
    title='Leave-One-Out evaluation: Hit Rate, Precision, NDCG @ K ∈ {5, 10, 20}',
    height=500,
    template='plotly_white',
    barmode='group',
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=1),
)
fig_loo.write_html(ARTIFACTS_DIR / 'week10_loo_evaluation_comparison.html')
fig_loo.write_image(ARTIFACTS_DIR / 'week10_loo_evaluation_comparison.png', scale=2)
fig_loo.show()


LOO evaluation results:


'           system  k  hit_rate_at_k  precision_at_k  ndcg_at_k  n_queries\npopularity_global  5         0.0287          0.0057     0.0180      10000\npopularity_global 10         0.0465          0.0047     0.0238      10000\npopularity_global 20         0.0767          0.0038     0.0313      10000\n   content_cosine  5         0.0219          0.0044     0.0139       9885\n   content_cosine 10         0.0368          0.0037     0.0187       9885\n   content_cosine 20         0.0563          0.0028     0.0236       9885\nsvd_collaborative  5         0.0515          0.0103     0.0343       9885\nsvd_collaborative 10         0.0795          0.0080     0.0432       9885\nsvd_collaborative 20         0.1221          0.0061     0.0538       9885'

### Save updated evaluation summary

In [17]:
# --- Actualizar eval_summary con resultados LOO ---
# Re-leer el summary existente y añadir la sección LOO

with open(ARTIFACTS_DIR / 'week10_evaluation_summary.json') as f:
    eval_summary_updated = json.load(f)

eval_summary_updated['loo_evaluation'] = {
    'evaluation_type': 'leave_one_out_user_history',
    'split_criterion': 'last_item_by_timestamp',
    'min_ratings_per_user': 5,
    'sample_users': LOO_SAMPLE_USERS,
    'random_seed': LOO_SEED,
    'k_values': K_VALUES_LOO,
    'recall_note': (
        'En LOO, Recall@K = Hit Rate@K porque hay exactamente 1 ítem de test '
        'por usuario. El Recall@K de la evaluación por género (secciones 1–10) '
        'tenía denominador igual al pool candidato del sistema — no al universo '
        'real — lo que sobreestimaba la métrica. Queda documentado en recall_at_k().'
    ),
    'results': {
        sys_name: {
            f'k_{k}': {
                'hit_rate':  round(hits[k] / n_q, 6),
                'precision': round(prec[k] / n_q, 6),
                'ndcg':      round(ndcg[k] / n_q, 6),
            }
            for k in K_VALUES_LOO
        }
        for sys_name, hits, prec, ndcg, n_q in [
            ('popularity_global', loo_pop_hits, loo_pop_prec, loo_pop_ndcg, loo_n_users),
            ('content_cosine',    loo_cb_hits,  loo_cb_prec,  loo_cb_ndcg,  loo_cb_n),
            ('svd_collaborative', loo_svd_hits, loo_svd_prec, loo_svd_ndcg, loo_svd_n),
        ]
    },
    'artifacts': [
        'week10_loo_evaluation_results.csv',
        'week10_loo_evaluation_comparison.html',
        'week10_loo_evaluation_comparison.png',
    ],
}

with open(ARTIFACTS_DIR / 'week10_evaluation_summary.json', 'w') as f:
    json.dump(eval_summary_updated, f, indent=2)

print('✓ week10_evaluation_summary.json actualizado con sección LOO.')
print(json.dumps(eval_summary_updated['loo_evaluation'], indent=2))


✓ week10_evaluation_summary.json actualizado con sección LOO.
{
  "evaluation_type": "leave_one_out_user_history",
  "split_criterion": "last_item_by_timestamp",
  "min_ratings_per_user": 5,
  "sample_users": 10000,
  "random_seed": 42,
  "k_values": [
    5,
    10,
    20
  ],
  "recall_note": "En LOO, Recall@K = Hit Rate@K porque hay exactamente 1 \u00edtem de test por usuario. El Recall@K de la evaluaci\u00f3n por g\u00e9nero (secciones 1\u201310) ten\u00eda denominador igual al pool candidato del sistema \u2014 no al universo real \u2014 lo que sobreestimaba la m\u00e9trica. Queda documentado en recall_at_k().",
  "results": {
    "popularity_global": {
      "k_5": {
        "hit_rate": 0.0287,
        "precision": 0.00574,
        "ndcg": 0.018044
      },
      "k_10": {
        "hit_rate": 0.0465,
        "precision": 0.00465,
        "ndcg": 0.023755
      },
      "k_20": {
        "hit_rate": 0.0767,
        "precision": 0.003835,
        "ndcg": 0.031293
      }
    },
   